# 11 — Silver Installments Payments

**Credit Risk Intelligence Platform** — Camada Silver

Este notebook transforma a tabela Bronze `credit_risk.bronze.installments_payments` em uma tabela Silver tratada, padronizada e preparada para análise e Machine Learning.

## Pipeline

```
credit_risk.bronze.installments_payments  →  credit_risk.silver.installments_payments
```

## Sobre a tabela installments_payments

A tabela `installments_payments` contém o histórico de pagamentos de parcelas de créditos anteriores do cliente no Home Credit. Cada registro representa um pagamento de parcela, identificado por `SK_ID_PREV`, com o número da parcela (`NUM_INSTALMENT_NUMBER`) e a versão do plano (`NUM_INSTALMENT_VERSION`).

**Importante**: A mesma combinação `(SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER, DAYS_INSTALMENT)` pode ter múltiplos registros — representam **pagamentos parciais múltiplos** para o mesmo parcelamento. Estes duplicatas são legítimas e NÃO são removidas.

## Transformações aplicadas

1. **Remoção de metadados Bronze** — colunas `_ingestion_timestamp` e `_source_file`
2. **Preservação de NULLs** — `DAYS_ENTRY_PAYMENT` e `AMT_PAYMENT` (0,02%) mantidos como NULL (significado: sem pagamento registrado)
3. **Flags de validação** — pagamento ausente, parcela zero
4. **Colunas de controle** — timestamp, versão, origem, hash
5. **Auditoria** — registro completo da transformação

## Regras

> A Bronze **NÃO é modificada**. Todas as transformações criam novas tabelas Silver.
> Nenhum registro é removido — as duplicatas na chave lógica são pagamentos parciais legítimos.
> Nenhuma agregação por cliente é realizada.
> Nenhuma feature de ML é criada (payment_delay, late_payment_flag, payment_ratio, etc. — reservadas para a camada Gold).
> Valores negativos em `DAYS_*` são **preservados** — representam dias relativos no Home Credit.

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração, Imports e Parâmetros
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Parâmetros do pipeline
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "silver_v1.0"
NOTEBOOK_NAME = "11_silver_installments_payments"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"silver_inst_pay_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)

# ----------------------------------------------------------------------------
# Tabelas de origem (Bronze) e destino (Silver)
# ----------------------------------------------------------------------------
BRONZE_TABLE = "credit_risk.bronze.installments_payments"
SILVER_TABLE = "credit_risk.silver.installments_payments"
AUDIT_TABLE = "credit_risk.silver.audit_transformation"

# Tabelas Silver para integridade referencial
SILVER_PREV_APP = "credit_risk.silver.previous_application"
SILVER_APP_TRAIN = "credit_risk.silver.application_train"
SILVER_APP_TEST = "credit_risk.silver.application_test"

# Colunas de metadados Bronze a remover na Silver
BRONZE_META_COLS = ["_ingestion_timestamp", "_source_file"]

# ----------------------------------------------------------------------------
# Criar schema Silver se não existir
# ----------------------------------------------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk.silver")
print(f"Schema credit_risk.silver verificado/criado.")

# ----------------------------------------------------------------------------
# Dicionário para registrar transformações aplicadas (para auditoria)
# ----------------------------------------------------------------------------
TRANSFORMATION_LOG = []

def log_transform(table_name, step, description, records_affected=0):
    """Registra uma transformação aplicada para auditoria."""
    TRANSFORMATION_LOG.append({
        "table": table_name,
        "step": step,
        "description": description,
        "records_affected": records_affected,
    })

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline Version: {PIPELINE_VERSION}")

In [0]:
# ============================================================================
# CÉLULA 2 — Leitura da Bronze e Inspeção do Schema
# ============================================================================
# Carrega o DataFrame Bronze (sem modificá-lo) e inspeciona o schema real.

df_ip_bronze = spark.table(BRONZE_TABLE)

bronze_row_count = df_ip_bronze.count()
bronze_col_count = len(df_ip_bronze.columns)

print("=" * 70)
print("INSPEÇÃO INICIAL — BRONZE")
print("=" * 70)
print(f"\n📊 {BRONZE_TABLE}")
print(f"   Registros: {bronze_row_count:,}")
print(f"   Colunas: {bronze_col_count}")

# Schema detalhado
sep = "─" * 70
print(f"\n{sep}")
print("SCHEMA — installments_payments (tipos e nullable)")
print(sep)
for field in df_ip_bronze.schema.fields:
    print(f"   {field.name:<30} {field.dataType.simpleString():<12} nullable={field.nullable}")

# Verificar colunas-chave esperadas
print(f"\n{sep}")
print("COLUNAS-CHAVE")
print(sep)
for c in ["SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]:
    if c in df_ip_bronze.columns:
        print(f"   ✅ {c}: presente")
    else:
        print(f"   ❌ {c}: AUSENTE")

print("\n✅ Leitura da Bronze concluída!")

In [0]:
# ============================================================================
# CÉLULA 3 — Data Quality Inicial (Bronze)
# ============================================================================
# Análise de completude (NULLs), valores distintos e estatísticas básicas.

sep = "─" * 70

# ----------------------------------------------------------------------------
# NULLs por coluna
# ----------------------------------------------------------------------------
null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_ip_bronze.columns]
null_row = df_ip_bronze.agg(*null_exprs).collect()[0]
null_pairs = [(c, null_row[c]) for c in df_ip_bronze.columns if null_row[c] and null_row[c] > 0]
null_pairs.sort(key=lambda x: x[1], reverse=True)

print(sep)
print(f"NULLs POR COLUNA — {BRONZE_TABLE} ({len(null_pairs)} cols com nulls)")
print(sep)
for c, n in null_pairs:
    pct = n / bronze_row_count * 100
    print(f"   {c:<30} {n:>10,}  ({pct:.2f}%)")
if not null_pairs:
    print("   Nenhuma coluna com NULLs")

# ----------------------------------------------------------------------------
# Valores distintos por coluna
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VALORES DISTINCTOS")
print(sep)
for c in df_ip_bronze.columns:
    d = df_ip_bronze.select(c).distinct().count()
    print(f"   {c:<30} {d:>10,}")

# ----------------------------------------------------------------------------
# Estatísticas numéricas
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("ESTATÍSTICAS NUMÉRICAS (min, max, mean, mediana, stddev)")
print(sep)
numeric_inspect = [
    "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT"
]
for c in numeric_inspect:
    if c in df_ip_bronze.columns:
        stats = df_ip_bronze.select(c).summary("min", "max", "mean", "50%", "stddev").collect()
        null_cnt = df_ip_bronze.filter(F.col(c).isNull()).count()
        neg_cnt = df_ip_bronze.filter(F.col(c) < 0).count()
        zero_cnt = df_ip_bronze.filter(F.col(c) == 0).count()
        print(f"\n   {c}: (null={null_cnt:,}, neg={neg_cnt:,}, zero={zero_cnt:,})")
        for s in stats:
            print(f"      {s['summary']:<10}: {s[c]}")

print("\n✅ Data Quality inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Validação de Identificadores e Duplicidades
# ============================================================================
# Valida chaves e analisa duplicidades. A chave lógica nao é única —
# múltiplos pagamentos parciais para o mesmo parcelamento são legítimos.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO DE IDENTIFICADORES E DUPLICIDADES")
print("=" * 70)

# ----------------------------------------------------------------------------
# Chaves individuais
# ----------------------------------------------------------------------------
for c in ["SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"]:
    if c in df_ip_bronze.columns:
        null_cnt = df_ip_bronze.filter(F.col(c).isNull()).count()
        distinct_cnt = df_ip_bronze.select(c).distinct().count()
        print(f"\n   {c}:")
        print(f"      NULL: {null_cnt}")
        print(f"      Distinct: {distinct_cnt:,}")

# ----------------------------------------------------------------------------
# Duplicidades
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("ANÁLISE DE DUPLICIDADES")
print(sep)

# Duplicidade completa
full_dups = bronze_row_count - df_ip_bronze.dropDuplicates().count()
print(f"\n   Duplicidade completa: {full_dups} linhas totalmente duplicadas")

# Duplicidade por chave lógica (SK_ID_PREV + NUM_INSTALMENT_VERSION + NUM_INSTALMENT_NUMBER)
logical_key = ["SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"]
logical_dups = bronze_row_count - df_ip_bronze.select(*logical_key).distinct().count()
print(f"   Duplicidade por ({' + '.join(logical_key)}): {logical_dups:,}")

# Adicionando DAYS_INSTALMENT
key_with_days = logical_key + ["DAYS_INSTALMENT"]
dups_with_days = bronze_row_count - df_ip_bronze.select(*key_with_days).distinct().count()
print(f"   Duplicidade por ({' + '.join(key_with_days)}): {dups_with_days:,}")

# Adicionando DAYS_ENTRY_PAYMENT (chave completa)
key_full = key_with_days + ["DAYS_ENTRY_PAYMENT"]
dups_full_key = bronze_row_count - df_ip_bronze.select(*key_full).distinct().count()
print(f"   Duplicidade por ({' + '.join(key_full)}): {dups_full_key:,}")

# ----------------------------------------------------------------------------
# Investigação das duplicatas
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("INVESTIGAÇÃO DAS DUPLICATAS NA CHAVE LÓGICA")
print(sep)

dup_groups = df_ip_bronze.groupBy(key_with_days).agg(
    F.count("*").alias("n_records")
).filter(F.col("n_records") > 1)

dup_group_count = dup_groups.count()
total_dup_records = dup_groups.agg(F.sum("n_records").alias("total")).collect()[0]["total"]

print(f"\n   Grupos com duplicatas (>1 registro): {dup_group_count:,}")
print(f"   Total de registros nesses grupos: {total_dup_records:,}")
print(f"   Registros extras (duplicatas): {total_dup_records - dup_group_count:,}")

# Distribuição de registros por grupo
print(f"\n   Distribuição de registros por grupo:")
dup_dist = dup_groups.groupBy("n_records").count().orderBy("n_records").collect()
for r in dup_dist:
    print(f"      {r['n_records']} registros: {r['count']:,} grupos")

# ----------------------------------------------------------------------------
# Justificativa: são pagamentos parciais múltiplos
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("JUSTIFICATIVA — PAGAMENTOS PARCIAIS MÚLTIPLOS")
print(sep)
print(f"\n   As duplicatas na chave lógica representam múltiplos pagamentos")
print(f"   para o mesmo parcelamento (mesma versão, número e data de vencimento).")
print(f"   Cada registro tem um DAYS_ENTRY_PAYMENT diferente (data do pagamento)")
print(f"   e um AMT_PAYMENT parcial que somam aproximadamente o AMT_INSTALMENT.")
print(f"\n   → DECISÃO: NÃO remover duplicatas — são pagamentos legítimos.")
print(f"   → Preservar todos os registros para manter o histórico completo.")

# Distribuição de registros por SK_ID_PREV
print(f"\n{sep}")
print("DISTRIBUIÇÃO DE REGISTROS POR SK_ID_PREV")
print(sep)

per_prev_stats = df_ip_bronze.groupBy("SK_ID_PREV").count().select("count")
summary = per_prev_stats.summary("min", "max", "mean", "50%").collect()
for s in summary:
    print(f"   {s['summary']:<10}: {s['count']}")

print("\n✅ Validação de identificadores e duplicidades concluída!")

In [0]:
# ============================================================================
# CÉLULA 5 — Validação de Campos de Parcelamento
# ============================================================================
# Inspeciona NUM_INSTALMENT_VERSION e NUM_INSTALMENT_NUMBER.
# Valida negativos, zeros, NULLs e valores extremos.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO DE CAMPOS DE PARCELAMENTO")
print("=" * 70)

# ----------------------------------------------------------------------------
# NUM_INSTALMENT_VERSION
# ----------------------------------------------------------------------------
print(f"\n📊 NUM_INSTALMENT_VERSION:")
ver_null = df_ip_bronze.filter(F.col("NUM_INSTALMENT_VERSION").isNull()).count()
ver_neg = df_ip_bronze.filter(F.col("NUM_INSTALMENT_VERSION") < 0).count()
ver_zero = df_ip_bronze.filter(F.col("NUM_INSTALMENT_VERSION") == 0).count()
ver_distinct = df_ip_bronze.select("NUM_INSTALMENT_VERSION").distinct().count()
print(f"   NULL: {ver_null}")
print(f"   Negativos: {ver_neg}")
print(f"   Zeros: {ver_zero:,} ({ver_zero/bronze_row_count*100:.2f}%)")
print(f"   Distinct: {ver_distinct}")

# Top 10 valores mais frequentes
print(f"\n   Top 10 valores mais frequentes:")
ver_dist2 = df_ip_bronze.groupBy("NUM_INSTALMENT_VERSION").count().orderBy(F.desc("count")).limit(10).collect()
for r in ver_dist2:
    pct = r['count'] / bronze_row_count * 100
    print(f"      {r['NUM_INSTALMENT_VERSION']:<10} {r['count']:>12,} ({pct:.2f}%)")

# ----------------------------------------------------------------------------
# NUM_INSTALMENT_NUMBER
# ----------------------------------------------------------------------------
print(f"\n📊 NUM_INSTALMENT_NUMBER:")
num_null = df_ip_bronze.filter(F.col("NUM_INSTALMENT_NUMBER").isNull()).count()
num_neg = df_ip_bronze.filter(F.col("NUM_INSTALMENT_NUMBER") < 0).count()
num_zero = df_ip_bronze.filter(F.col("NUM_INSTALMENT_NUMBER") == 0).count()
num_distinct = df_ip_bronze.select("NUM_INSTALMENT_NUMBER").distinct().count()
print(f"   NULL: {num_null}")
print(f"   Negativos: {num_neg}")
print(f"   Zeros: {num_zero}")
print(f"   Distinct: {num_distinct}")

# Estatísticas
num_stats = df_ip_bronze.select("NUM_INSTALMENT_NUMBER").summary("min", "max", "mean", "50%").collect()
print(f"\n   Estatísticas:")
for s in num_stats:
    print(f"      {s['summary']:<10}: {s['NUM_INSTALMENT_NUMBER']}")

# Top 10 valores mais frequentes
print(f"\n   Top 10 valores mais frequentes:")
num_dist2 = df_ip_bronze.groupBy("NUM_INSTALMENT_NUMBER").count().orderBy(F.desc("count")).limit(10).collect()
for r in num_dist2:
    pct = r['count'] / bronze_row_count * 100
    print(f"      {r['NUM_INSTALMENT_NUMBER']:<10} {r['count']:>12,} ({pct:.2f}%)")

# ----------------------------------------------------------------------------
# Validação cruzada
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VALIDAÇÃO CRUZADA")
print(sep)

if num_zero == 0 and num_neg == 0:
    print(f"   ✅ NUM_INSTALMENT_NUMBER: todos >= 1 (sem zeros ou negativos)")
else:
    print(f"   ⚠️ NUM_INSTALMENT_NUMBER: {num_neg} negativos, {num_zero} zeros")

print(f"   ℹ️ NUM_INSTALMENT_VERSION: 0 é válido (versão inicial) — {ver_zero:,} registros")

print("\n✅ Validação de campos de parcelamento concluída!")

In [0]:
# ============================================================================
# CÉLULA 6 — Validação de Campos Temporais
# ============================================================================
# Inspeciona DAYS_INSTALMENT e DAYS_ENTRY_PAYMENT.
# Valores negativos são ESPERADOS — representam dias relativos à aplicação.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO DE CAMPOS TEMPORAIS")
print("=" * 70)

# ----------------------------------------------------------------------------
# DAYS_INSTALMENT (data de vencimento da parcela)
# ----------------------------------------------------------------------------
print(f"\n📊 DAYS_INSTALMENT (dia de vencimento):")
di_null = df_ip_bronze.filter(F.col("DAYS_INSTALMENT").isNull()).count()
di_neg = df_ip_bronze.filter(F.col("DAYS_INSTALMENT") < 0).count()
di_pos = df_ip_bronze.filter(F.col("DAYS_INSTALMENT") >= 0).count()
print(f"   NULL: {di_null}")
print(f"   Negativos: {di_neg:,} ({di_neg/bronze_row_count*100:.2f}%)")
print(f"   Não-negativos: {di_pos} (esperado: 0)")

di_stats = df_ip_bronze.select("DAYS_INSTALMENT").summary("min", "max", "mean", "50%", "stddev").collect()
print(f"\n   Estatísticas:")
for s in di_stats:
    print(f"      {s['summary']:<10}: {s['DAYS_INSTALMENT']}")

# Distribuição por faixas
print(f"\n   Distribuição por faixas:")
bins = [(-2922, -2500), (-2499, -2000), (-1999, -1500), (-1499, -1000), (-999, -500), (-499, -100), (-99, -1)]
for lo, hi in bins:
    cnt = df_ip_bronze.filter((F.col("DAYS_INSTALMENT") >= lo) & (F.col("DAYS_INSTALMENT") <= hi)).count()
    print(f"      {lo:>5} a {hi:>5}: {cnt:>12,} ({cnt/bronze_row_count*100:.2f}%)")

# ----------------------------------------------------------------------------
# DAYS_ENTRY_PAYMENT (dia em que o pagamento foi efetuado)
# ----------------------------------------------------------------------------
print(f"\n{'=' * 70}")
print(f"📊 DAYS_ENTRY_PAYMENT (dia do pagamento):")
dep_null = df_ip_bronze.filter(F.col("DAYS_ENTRY_PAYMENT").isNull()).count()
dep_neg = df_ip_bronze.filter(F.col("DAYS_ENTRY_PAYMENT") < 0).count()
dep_pos = df_ip_bronze.filter(F.col("DAYS_ENTRY_PAYMENT") >= 0).count()
print(f"   NULL: {dep_null:,} ({dep_null/bronze_row_count*100:.4f}%)")
print(f"   Negativos: {dep_neg:,} ({dep_neg/bronze_row_count*100:.2f}%)")
print(f"   Não-negativos: {dep_pos} (esperado: 0)")

dep_stats = df_ip_bronze.select("DAYS_ENTRY_PAYMENT").summary("min", "max", "mean", "50%", "stddev").collect()
print(f"\n   Estatísticas:")
for s in dep_stats:
    print(f"      {s['summary']:<10}: {s['DAYS_ENTRY_PAYMENT']}")

# ----------------------------------------------------------------------------
# Análise da relação entre DAYS_ENTRY_PAYMENT e DAYS_INSTALMENT
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("RELAÇÃO DAYS_ENTRY_PAYMENT vs DAYS_INSTALMENT")
print(sep)

print(f"\n   Registros sem pagamento (DAYS_ENTRY_PAYMENT NULL): {dep_null:,}")

before = df_ip_bronze.filter(
    F.col("DAYS_ENTRY_PAYMENT").isNotNull() &
    (F.col("DAYS_ENTRY_PAYMENT") < F.col("DAYS_INSTALMENT"))
).count()
print(f"   Pagamento antes do vencimento: {before:,} ({before/bronze_row_count*100:.2f}%)")

same_day = df_ip_bronze.filter(
    F.col("DAYS_ENTRY_PAYMENT").isNotNull() &
    (F.col("DAYS_ENTRY_PAYMENT") == F.col("DAYS_INSTALMENT"))
).count()
print(f"   Pagamento no vencimento (igual): {same_day:,} ({same_day/bronze_row_count*100:.2f}%)")

after = df_ip_bronze.filter(
    F.col("DAYS_ENTRY_PAYMENT").isNotNull() &
    (F.col("DAYS_ENTRY_PAYMENT") > F.col("DAYS_INSTALMENT"))
).count()
print(f"   Pagamento depois do vencimento: {after:,} ({after/bronze_row_count*100:.2f}%)")

print(f"\n   ℹ️ Não criar feature de atraso neste notebook (reservado para Gold)")

print("\n✅ Validação de campos temporais concluída!")

In [0]:
# ============================================================================
# CÉLULA 7 — Validação de Campos Financeiros
# ============================================================================
# Inspeciona AMT_INSTALMENT e AMT_PAYMENT.
# Verifica negativos, zeros, NULLs, valores extremos.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO DE CAMPOS FINANCEIROS")
print("=" * 70)

for c in ["AMT_INSTALMENT", "AMT_PAYMENT"]:
    null_cnt = df_ip_bronze.filter(F.col(c).isNull()).count()
    neg_cnt = df_ip_bronze.filter(F.col(c) < 0).count()
    zero_cnt = df_ip_bronze.filter(F.col(c) == 0).count()

    stats = df_ip_bronze.filter(F.col(c).isNotNull()).select(c).summary(
        "min", "max", "mean", "50%", "stddev"
    ).collect()

    print(f"\n📊 {c}:")
    print(f"   NULL: {null_cnt:,} ({null_cnt/bronze_row_count*100:.4f}%)")
    print(f"   Negativos: {neg_cnt}")
    print(f"   Zeros: {zero_cnt:,} ({zero_cnt/bronze_row_count*100:.4f}%)")
    print(f"\n   Estatísticas:")
    for s in stats:
        print(f"      {s['summary']:<10}: {s[c]}")

    pcts = df_ip_bronze.filter(F.col(c).isNotNull()).select(c).summary(
        "25%", "75%", "90%", "95%", "99%"
    ).collect()
    print(f"\n   Percentis:")
    for s in pcts:
        print(f"      {s['summary']:<10}: {s[c]}")

# ----------------------------------------------------------------------------
# Verificação de valores extremos
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VALORES EXTREMOS")
print(sep)

instalment_extreme = df_ip_bronze.filter(F.col("AMT_INSTALMENT") > 1000000).count()
print(f"   AMT_INSTALMENT > 1.000.000: {instalment_extreme:,}")

payment_extreme = df_ip_bronze.filter(F.col("AMT_PAYMENT") > 1000000).count()
print(f"   AMT_PAYMENT > 1.000.000: {payment_extreme:,}")

instalment_zero = df_ip_bronze.filter(F.col("AMT_INSTALMENT") == 0).count()
print(f"   AMT_INSTALMENT = 0: {instalment_zero:,} (parcela sem valor)")

payment_zero = df_ip_bronze.filter(F.col("AMT_PAYMENT") == 0).count()
print(f"   AMT_PAYMENT = 0: {payment_zero:,} (pagamento zero)")

print("\n✅ Validação de campos financeiros concluída!")

In [0]:
# ============================================================================
# CÉLULA 8 — Consistência de Pagamentos
# ============================================================================
# Avalia regras de consistência entre parcelamento e pagamento.
# NÃO cria features de ML (atraso, ratio, etc.) — apenas identifica inconsistências.

sep = "─" * 70
print("=" * 70)
print("CONSISTÊNCIA DE PAGAMENTOS")
print("=" * 70)

# ----------------------------------------------------------------------------
# Regras de consistência
# ----------------------------------------------------------------------------
rules = [
    ("AMT_INSTALMENT negativo",
     df_ip_bronze.filter(F.col("AMT_INSTALMENT") < 0).count()),
    ("AMT_PAYMENT negativo",
     df_ip_bronze.filter(F.col("AMT_PAYMENT") < 0).count()),
    ("AMT_INSTALMENT = 0 (parcela zero)",
     df_ip_bronze.filter(F.col("AMT_INSTALMENT") == 0).count()),
    ("AMT_PAYMENT = 0 (pagamento zero)",
     df_ip_bronze.filter(F.col("AMT_PAYMENT") == 0).count()),
    ("AMT_PAYMENT NULL (sem pagamento)",
     df_ip_bronze.filter(F.col("AMT_PAYMENT").isNull()).count()),
    ("DAYS_ENTRY_PAYMENT NULL (sem data de pagamento)",
     df_ip_bronze.filter(F.col("DAYS_ENTRY_PAYMENT").isNull()).count()),
    ("DAYS_INSTALMENT nao-negativo (anomalia)",
     df_ip_bronze.filter(F.col("DAYS_INSTALMENT") >= 0).count()),
]

print(f"\n   {'Regra':<50} {'Afetados':>10} {'%':>8}")
print(f"   {sep}")
for rule, affected in rules:
    pct = affected / bronze_row_count * 100
    print(f"   {rule:<50} {affected:>10,} {pct:>7.4f}%")

# ----------------------------------------------------------------------------
# Pagamento maior que parcela (situação legítima — não é erro)
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("PAGAMENTO MAIOR QUE PARCELA")
print(sep)

overpay = df_ip_bronze.filter(
    F.col("AMT_PAYMENT").isNotNull() &
    (F.col("AMT_PAYMENT") > F.col("AMT_INSTALMENT"))
).count()
print(f"   Registros com AMT_PAYMENT > AMT_INSTALMENT: {overpay:,} ({overpay/bronze_row_count*100:.2f}%)")
print(f"   ℹ️ Pagamento maior que parcela é LEGÍTIMO — pode representar:")
print(f"      - Pagamento antecipado de parcelas futuras")
print(f"      - Ajustes de cambio/juros")
print(f"      - Pagamento consolidado")
print(f"   → NÃO excluir estes registros")

# ----------------------------------------------------------------------------
# Resumo de inconsistências
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("RESUMO DE INCONSISTÊNCIAS (WARNING — não remover)")
print(sep)
print(f"\n   Registros com AMT_INSTALMENT = 0: {df_ip_bronze.filter(F.col('AMT_INSTALMENT') == 0).count():,}")
print(f"   Registros sem pagamento (NULL): {df_ip_bronze.filter(F.col('AMT_PAYMENT').isNull()).count():,}")
print(f"   Registros com pagamento > parcela: {overpay:,}")
print(f"\n   ⚠️ WARNING: Todas as inconsistências são preservadas.")
print(f"      Features de atraso/ratio serao criadas na camada Gold.")

print("\n✅ Consistência de pagamentos validada!")

In [0]:
# ============================================================================
# CÉLULA 9 — Integridade Referencial
# ============================================================================
# Valida SK_ID_PREV contra silver.previous_application e
# SK_ID_CURR contra silver.application_train + application_test.

sep = "─" * 70
print("=" * 70)
print("INTEGRIDADE REFERENCIAL")
print("=" * 70)

# ----------------------------------------------------------------------------
# SK_ID_PREV vs silver.previous_application
# ----------------------------------------------------------------------------
print(f"\n📊 installments_payments.SK_ID_PREV vs silver.previous_application:")
try:
    prev_app_sk = spark.table(SILVER_PREV_APP).select("SK_ID_PREV").distinct()
    ip_sk_prev = df_ip_bronze.select("SK_ID_PREV").distinct()
    ip_sk_prev_count = ip_sk_prev.count()

    matched_prev = ip_sk_prev.join(prev_app_sk, "SK_ID_PREV", "inner").count()
    unmatched_prev = ip_sk_prev_count - matched_prev

    print(f"   SK_ID_PREV distintos na installments: {ip_sk_prev_count:,}")
    print(f"   Correspondidos: {matched_prev:,} ({matched_prev/ip_sk_prev_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_prev:,} ({unmatched_prev/ip_sk_prev_count*100:.2f}%)")
    if unmatched_prev == 0:
        print(f"   → ✅ Todos os SK_ID_PREV têm correspondência")
    else:
        print(f"   → ⚠️ {unmatched_prev} SK_ID_PREV sem correspondência (registros preservados)")
except Exception as e:
    print(f"   ⚠️ Não foi possível verificar: {e}")

# ----------------------------------------------------------------------------
# SK_ID_CURR vs silver.application_train + application_test
# ----------------------------------------------------------------------------
print(f"\n📊 installments_payments.SK_ID_CURR vs silver.application (train + test):")
try:
    app_train_sk = spark.table(SILVER_APP_TRAIN).select("SK_ID_CURR").distinct()
    app_test_sk = spark.table(SILVER_APP_TEST).select("SK_ID_CURR").distinct()
    all_app_sk = app_train_sk.union(app_test_sk).distinct()

    ip_sk_curr = df_ip_bronze.select("SK_ID_CURR").distinct()
    ip_sk_curr_count = ip_sk_curr.count()

    matched_curr = ip_sk_curr.join(all_app_sk, "SK_ID_CURR", "inner").count()
    unmatched_curr = ip_sk_curr_count - matched_curr

    print(f"   SK_ID_CURR distintos na installments: {ip_sk_curr_count:,}")
    print(f"   Correspondidos: {matched_curr:,} ({matched_curr/ip_sk_curr_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_curr:,} ({unmatched_curr/ip_sk_curr_count*100:.2f}%)")
    if unmatched_curr == 0:
        print(f"   → ✅ Todos os SK_ID_CURR têm correspondência")
    else:
        print(f"   → ⚠️ {unmatched_curr} SK_ID_CURR sem correspondência (registros preservados)")
except Exception as e:
    print(f"   ⚠️ Não foi possível verificar: {e}")

print("\n✅ Integridade referencial validada!")

In [0]:
# ============================================================================
# CÉLULA 10 — Funções de Transformação Reutilizáveis
# ============================================================================
# Funções modulares aplicadas na transformação Bronze → Silver.

def remove_bronze_metadata(df, table_name):
    """Remove colunas de metadados da Bronze."""
    cols_to_drop = [c for c in BRONZE_META_COLS if c in df.columns]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        log_transform(table_name, "remove_metadata", f"Removidas colunas Bronze: {cols_to_drop}")
    return df


def preserve_nulls(df, table_name, row_count):
    """Documenta NULLs preservados.
    DAYS_ENTRY_PAYMENT e AMT_PAYMENT têm 2.905 NULLs (0,02%) —
    significam que o pagamento não foi registrado (ausência de pagamento).
    NULLs são preservados sem substituição por zero."""
    null_cols = ["DAYS_ENTRY_PAYMENT", "AMT_PAYMENT"]
    for c in null_cols:
        if c in df.columns:
            null_cnt = df.filter(F.col(c).isNull()).count()
            if null_cnt > 0:
                log_transform(table_name, "preserve_null",
                    f"{c}: {null_cnt} NULLs preservados (sem pagamento registrado)", null_cnt)
                print(f"   ℹ️ {c}: {null_cnt} NULLs preservados")
    return df


def add_validation_flags(df, table_name, row_count):
    """Adiciona flags de validação para dados quality.
    NÃO cria features de ML (atraso, ratio, etc.) — apenas flags de DQ."""
    flags_added = []

    # FLAG_PAYMENT_MISSING: pagamento ausente (AMT_PAYMENT IS NULL)
    if "AMT_PAYMENT" in df.columns:
        missing_cnt = df.filter(F.col("AMT_PAYMENT").isNull()).count()
        df = df.withColumn("FLAG_PAYMENT_MISSING",
            F.when(F.col("AMT_PAYMENT").isNull(), 1).otherwise(0))
        flags_added.append(("FLAG_PAYMENT_MISSING", missing_cnt))

    # FLAG_AMT_INSTALMENT_ZERO: parcela com valor zero
    if "AMT_INSTALMENT" in df.columns:
        zero_cnt = df.filter(F.col("AMT_INSTALMENT") == 0).count()
        df = df.withColumn("FLAG_AMT_INSTALMENT_ZERO",
            F.when(F.col("AMT_INSTALMENT") == 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_INSTALMENT_ZERO", zero_cnt))

    # FLAG_AMT_PAYMENT_ZERO: pagamento zero
    if "AMT_PAYMENT" in df.columns:
        pay_zero_cnt = df.filter(F.col("AMT_PAYMENT") == 0).count()
        df = df.withColumn("FLAG_AMT_PAYMENT_ZERO",
            F.when(F.col("AMT_PAYMENT") == 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_PAYMENT_ZERO", pay_zero_cnt))

    for flag_name, affected_count in flags_added:
        log_transform(table_name, "validation_flag",
            f"{flag_name}: {affected_count} registros marcados", affected_count)
        print(f"   ✅ {flag_name}: {affected_count} registros")
    return df


def add_control_columns(df, source_table):
    """Adiciona colunas de controle técnicas da Silver."""
    df = df.withColumn("silver_processing_timestamp", F.current_timestamp())
    df = df.withColumn("silver_processing_date", F.current_date())
    df = df.withColumn("silver_pipeline_version", F.lit(PIPELINE_VERSION))
    df = df.withColumn("source_table", F.lit(source_table))

    data_cols = [c for c in df.columns if c not in [
        "silver_processing_timestamp", "silver_processing_date",
        "silver_pipeline_version", "source_table"
    ]]
    hash_expr = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in data_cols])
    df = df.withColumn("record_hash", F.md5(hash_expr))
    print(f"   ✅ Colunas de controle adicionadas (timestamp, date, version, source, hash)")
    return df


def apply_silver_transformations(df, table_name, source_table, row_count):
    """Aplica todas as transformações Silver em sequência."""
    print(f"\n{'─' * 60}")
    print(f"🔧 Transformando: {table_name}")
    print(f"{'─' * 60}")

    df = remove_bronze_metadata(df, table_name)
    df = preserve_nulls(df, table_name, row_count)
    df = add_validation_flags(df, table_name, row_count)
    df = add_control_columns(df, source_table)

    print(f"   ✅ Transformações concluídas para {table_name}")
    return df


print("✅ Funções de transformação definidas!")

In [0]:
# ============================================================================
# CÉLULA 11 — Execução das Transformações
# ============================================================================
# Aplica as transformações Silver na tabela installments_payments.
# O DataFrame Bronze original não é modificado.

EXEC_START = datetime.now(timezone.utc)

print("=" * 70)
print("TRANSFORMAÇÃO SILVER — installments_payments")
print("=" * 70)
transform_start = datetime.now(timezone.utc)

df_ip_silver = apply_silver_transformations(
    df_ip_bronze, SILVER_TABLE, BRONZE_TABLE, bronze_row_count
)

transform_end = datetime.now(timezone.utc)
transform_duration = (transform_end - transform_start).total_seconds()
silver_row_count = df_ip_silver.count()
silver_col_count = len(df_ip_silver.columns)

print(f"\n   Bronze: {bronze_row_count:,} rows x {bronze_col_count} cols")
print(f"   Silver: {silver_row_count:,} rows x {silver_col_count} cols")
print(f"   Duração: {transform_duration:.1f}s")

print(f"\n{'=' * 70}")
print(f"⏱️ Tempo total de transformação: {transform_duration:.1f}s")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 12 — Escrita da Tabela Silver (Delta Lake)
# ============================================================================
# Grava a tabela Silver usando mode("overwrite") com overwriteSchema.

print("=" * 70)
print("GRAVAÇÃO DA TABELA SILVER")
print("=" * 70)

print(f"\n📊 Gravando {SILVER_TABLE}...")
write_start = datetime.now(timezone.utc)

df_ip_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(SILVER_TABLE)

write_end = datetime.now(timezone.utc)
write_duration = (write_end - write_start).total_seconds()
print(f"   ✅ {SILVER_TABLE} gravada em {write_duration:.1f}s")
print(f"      Registros: {silver_row_count:,} | Colunas: {silver_col_count}")

EXEC_END = datetime.now(timezone.utc)
TOTAL_DURATION = (EXEC_END - EXEC_START).total_seconds()

print(f"\n{'=' * 70}")
print("✅ TABELA SILVER GRAVADA COM SUCESSO!")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 13 — Auditoria da Transformação
# ============================================================================
# Cria/atualiza a tabela credit_risk.silver.audit_transformation (append mode).
from pyspark.sql.types import (StructType, StructField, StringType,
    IntegerType, DoubleType, TimestampType, LongType)

audit_record = {
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "execution_id": EXECUTION_ID,
    "batch_id": BATCH_ID,
    "source_table": BRONZE_TABLE,
    "target_table": SILVER_TABLE,
    "source_row_count": bronze_row_count,
    "target_row_count": silver_row_count,
    "records_inserted": silver_row_count,
    "records_removed": 0,
    "records_changed": silver_row_count,
    "processing_duration_seconds": float(TOTAL_DURATION),
    "pipeline_version": PIPELINE_VERSION,
    "execution_status": "SUCCESS",
    "error_message": "",
}

audit_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("source_row_count", LongType(), True),
    StructField("target_row_count", LongType(), True),
    StructField("records_inserted", LongType(), True),
    StructField("records_removed", IntegerType(), True),
    StructField("records_changed", LongType(), True),
    StructField("processing_duration_seconds", DoubleType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("execution_status", StringType(), True),
    StructField("error_message", StringType(), True),
])

audit_df = spark.createDataFrame([audit_record], schema=audit_schema)

print(f"📊 Persistindo auditoria em {AUDIT_TABLE}...")
audit_df.write.mode("append").format("delta").saveAsTable(AUDIT_TABLE)

print(f"✅ Auditoria registrada: 1 registro em {AUDIT_TABLE}")
print("\nRegistros de auditoria (últimos 10):")
display(spark.table(AUDIT_TABLE).orderBy(F.col("execution_timestamp").desc()).limit(10))

In [0]:
# ============================================================================
# CÉLULA 14 — Data Quality Pós-Transformação (Bronze vs Silver)
# ============================================================================
# Compara métricas de qualidade antes (Bronze) e depois (Silver).

def compute_dq_metrics(df, table_name):
    """Computa métricas de DQ: row_count, col_count, null_count, duplicate_count."""
    row_count = df.count()
    col_count = len(df.columns)
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) for c in df.columns]
    total_nulls = df.agg(*null_exprs).collect()[0]
    null_sum = sum([total_nulls[i] for i in range(len(df.columns))])
    if "SK_ID_PREV" in df.columns and "NUM_INSTALMENT_VERSION" in df.columns:
        dup_count = row_count - df.select("SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER").distinct().count()
    else:
        dup_count = 0
    return {
        "table": table_name,
        "row_count": row_count,
        "col_count": col_count,
        "null_count": null_sum,
        "null_percentage": round(null_sum / (row_count * col_count) * 100, 4) if row_count > 0 else 0,
        "duplicate_count": dup_count,
    }

# Ler tabela Silver recém-criada
df_silver = spark.table(SILVER_TABLE)

sep = "─" * 75
print("=" * 70)
print("DATA QUALITY: BRONZE vs SILVER")
print("=" * 70)

bronze_m = compute_dq_metrics(df_ip_bronze, BRONZE_TABLE)
silver_m = compute_dq_metrics(df_silver, SILVER_TABLE)

print(f"\n📊 installments_payments")
print(f"{'Métrica':<30} {'Bronze':>15} {'Silver':>15} {'Delta':>15}")
print(sep)
for key in ["row_count", "col_count", "null_count", "null_percentage", "duplicate_count"]:
    b = bronze_m[key]
    s = silver_m[key]
    d = s - b
    print(f"{key:<30} {b:>15,} {s:>15,} {d:>+15,}")

# Verificações específicas
print(f"\n{sep}")
print("VERIFICAÇÕES ESPECÍFICAS")
print(sep)

# Colunas de controle presentes
control_cols = ["silver_processing_timestamp", "silver_processing_date",
                "silver_pipeline_version", "source_table", "record_hash"]
for c in control_cols:
    present = c in df_silver.columns
    print(f"   Coluna {c}: {'✅ presente' if present else '❌ ausente'}")

# Colunas Bronze removidas
print(f"\n   Colunas Bronze removidas:")
for c in BRONZE_META_COLS:
    present = c in df_silver.columns
    print(f"      {c}: {'❌ ainda presente' if present else '✅ removida'}")

# Flags de validação
validation_flags = [c for c in df_silver.columns if c.startswith("FLAG_")]
print(f"\n   Flags de validação ({len(validation_flags)}):")
for flag in validation_flags:
    count = df_silver.filter(F.col(flag) == 1).count()
    print(f"      {flag}: {count} registros")

# Chaves preservadas
print(f"\n   Chaves preservadas:")
for c in ["SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"]:
    if c in df_silver.columns:
        d = df_silver.select(c).distinct().count()
        print(f"      {c}: {d:,} distinct")

# NULLs preservados
print(f"\n   NULLs preservados:")
for c in ["DAYS_ENTRY_PAYMENT", "AMT_PAYMENT"]:
    if c in df_silver.columns:
        n = df_silver.filter(F.col(c).isNull()).count()
        print(f"      {c}: {n:,}")

# Valores negativos preservados
print(f"\n   Valores negativos preservados:")
for c in ["DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]:
    if c in df_silver.columns:
        n = df_silver.filter(F.col(c) < 0).count()
        print(f"      {c}: {n:,} negativos")

print("\n✅ Data Quality pós-transformação concluída!")

In [0]:
# ============================================================================
# CÉLULA 15 — Validação Final e Amostras
# ============================================================================
# Valida que a tabela Silver está correta e coerente com a Bronze.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO FINAL — TABELA SILVER")
print("=" * 70)

print(f"\n📊 {SILVER_TABLE}")
print(sep)

# Comparar row count
assert silver_row_count == bronze_row_count, \
    f"Row count mismatch: Bronze={bronze_row_count} vs Silver={silver_row_count}"
print(f"   ✅ Row count: {silver_row_count:,} (igual à Bronze)")

# Colunas
print(f"   Colunas Bronze: {bronze_col_count}")
print(f"   Colunas Silver: {silver_col_count}")
print(f"   Colunas adicionadas: {silver_col_count - bronze_col_count}")
print(f"     - Removidas: 2 (metadados Bronze)")
print(f"     - Adicionadas: 3 flags validação + 5 colunas controle")

# Estatísticas das colunas financeiras na Silver
print(f"\n{sep}")
print("ESTATÍSTICAS FINANCEIRAS (Silver)")
print(sep)
for c in ["AMT_INSTALMENT", "AMT_PAYMENT"]:
    if c in df_silver.columns:
        stats = df_silver.select(c).summary("min", "max", "mean", "50%", "stddev").collect()
        null_cnt = df_silver.filter(F.col(c).isNull()).count()
        print(f"\n   {c}: (null={null_cnt:,})")
        for s in stats:
            print(f"      {s['summary']:<10}: {s[c]}")

# Distribuição dos campos de dias
print(f"\n{sep}")
print("CAMPOS DE DIAS (Silver)")
print(sep)
for c in ["DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]:
    if c in df_silver.columns:
        stats = df_silver.select(c).summary("min", "max", "mean", "50%").collect()
        null_cnt = df_silver.filter(F.col(c).isNull()).count()
        print(f"\n   {c}: (null={null_cnt:,})")
        for s in stats:
            print(f"      {s['summary']:<10}: {s[c]}")

# Quantidade de NULLs
print(f"\n{sep}")
print("NULLs NA SILVER")
print(sep)
null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_silver.columns]
null_row = df_silver.agg(*null_exprs).collect()[0]
for c in df_silver.columns:
    n = null_row[c]
    if n and n > 0:
        print(f"   {c:<30} {n:>10,}")

# Registros com inconsistências
print(f"\n{sep}")
print("REGISTROS COM POSSÍVEIS INCONSISTÊNCIAS")
print(sep)
flag_cols = [c for c in df_silver.columns if c.startswith("FLAG_")]
for flag in flag_cols:
    count = df_silver.filter(F.col(flag) == 1).count()
    print(f"   {flag}: {count:,} registros")

# Amostra
print(f"\n{sep}")
print(f"AMOSTRA — {SILVER_TABLE} (primeiras 20 linhas)")
print(sep)

sample_cols = [
    "SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT",
    "FLAG_PAYMENT_MISSING", "FLAG_AMT_INSTALMENT_ZERO", "FLAG_AMT_PAYMENT_ZERO",
    "silver_processing_timestamp", "silver_pipeline_version",
    "source_table", "record_hash"
]
sample_cols = [c for c in sample_cols if c in df_silver.columns]
display(df_silver.select(*sample_cols).limit(20))

print("\n✅ Validação final concluída com sucesso!")

In [0]:
# ============================================================================
# CÉLULA 16 — Resumo Final da Execução
# ============================================================================
# Exibe um resumo completo da transformação.

print("=" * 60)
print("SILVER INSTALLMENTS PAYMENTS - RESUMO")
print("=" * 60)

print(f"\nOrigem:\n  {BRONZE_TABLE}")
print(f"\nDestino:\n  {SILVER_TABLE}")
print(f"\nRegistros Bronze:\n  {bronze_row_count:,}")
print(f"\nRegistros Silver:\n  {silver_row_count:,}")
print(f"\nRegistros removidos:\n  0")
print(f"\nRegistros alterados:\n  {silver_row_count:,} (transformações aplicadas)")
print(f"\nColunas:\n  Bronze: {bronze_col_count}")
print(f"  Silver: {silver_col_count}")

# NULLs preservados
nulls_preserved = sum(t["records_affected"] for t in TRANSFORMATION_LOG if t["step"] == "preserve_null")
print(f"\nNULLs preservados:\n  {nulls_preserved:,} (DAYS_ENTRY_PAYMENT + AMT_PAYMENT)")

# Duplicidades
print(f"\nDuplicidades identificadas:")
print(f"  Completa: 0")
print(f"  (SK_ID_PREV + VERSION + NUMBER + DAYS_INST): 653,483 (pagamentos parciais multiplos)")
print(f"\nDuplicidades removidas:\n  0 (legitimas — pagamentos parciais)")

# Anomalias identificadas
print(f"\nAnomalias identificadas:")
for t in TRANSFORMATION_LOG:
    if t["step"] == "validation_flag":
        print(f"  • {t['description']}")

# Transformações aplicadas
print(f"\nRegras aplicadas ({len(TRANSFORMATION_LOG)}):")
for t in TRANSFORMATION_LOG:
    print(f"  • {t['step']}: {t['description']}")

# Integridade referencial
print(f"\nIntegridade referencial:")
print(f"  SK_ID_PREV vs previous_application: verificada (registros preservados)")
print(f"  SK_ID_CURR vs application (train+test): verificada (registros preservados)")

# Warnings
warnings = [t for t in TRANSFORMATION_LOG if "WARNING" in t.get("description", "")]
print(f"\nWarnings:\n  {len(warnings)}")

print(f"\nStatus:\n  SUCCESS")
print(f"\nTempo:\n  {TOTAL_DURATION:.1f} segundos")
print(f"\n⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"\n{'=' * 60}")
print("✅ PIPELINE SILVER INSTALLMENTS PAYMENTS CONCLUÍDO COM SUCESSO!")
print(f"{'=' * 60}")

## Transformações Aplicadas — Documentação

### 1. Remoção de metadados Bronze
Colunas `_ingestion_timestamp` e `_source_file` removidas (substituídas por colunas de controle Silver).

### 2. Preservação de NULLs

2 colunas têm NULLs significativos, todos **preservados** sem substituição:

| Coluna | NULLs | % | Significado |
|--------|-------|---|------------|
| `DAYS_ENTRY_PAYMENT` | 2.905 | 0,02% | Sem pagamento registrado (ausência de pagamento) |
| `AMT_PAYMENT` | 2.905 | 0,02% | Sem pagamento registrado (ausência de pagamento) |

- Os 2.905 NULLs correspondem aos mesmos registros (sem pagamento = sem data de pagamento)
- Substituir NULLs por zero alteraria a semântica (zero = pagamento de valor zero, NULL = ausência de pagamento)

### 3. Duplicidades

| Tipo | Quantidade | Ação |
|------|------------|------|
| Linhas totalmente duplicadas | 0 | Nenhuma ação necessária |
| (SK_ID_PREV + VERSION + NUMBER) | 653.483 | **Preservadas** |
| (SK_ID_PREV + VERSION + NUMBER + DAYS_INST) | 653.483 | **Preservadas** |
| + DAYS_ENTRY_PAYMENT | 770 | **Preservadas** |

**Justificativa**: As 653.483 duplicatas na chave lógica são **pagamentos parciais múltiplos** para o mesmo parcelamento. Cada registro tem:
- Mesmo `SK_ID_PREV`, `NUM_INSTALMENT_VERSION`, `NUM_INSTALMENT_NUMBER`, `DAYS_INSTALMENT`
- Diferente `DAYS_ENTRY_PAYMENT` (data do pagamento)
- Diferente `AMT_PAYMENT` (valor parcial que soma ~`AMT_INSTALMENT`)

Distribuição: 629.210 grupos com 2 registros, 11.000 com 3, 578 com 4, etc.

### 4. Campos de Parcelamento

| Campo | Min | Max | Distinct | Observação |
|-------|-----|-----|----------|-----------|
| `NUM_INSTALMENT_VERSION` | 0 | 178 | 65 | 0 = versão inicial (30% dos registros) |
| `NUM_INSTALMENT_NUMBER` | 1 | 277 | 277 | Sem zeros ou negativos ✅ |

### 5. Campos Temporais

| Campo | Min | Max | Média | NULLs | Negativos |
|-------|-----|-----|-------|-------|-----------|
| `DAYS_INSTALMENT` | -2.922 | -1 | -1.042 | 0 | 100% ✅ |
| `DAYS_ENTRY_PAYMENT` | -4.921 | -1 | -1.051 | 2.905 | ~100% ✅ |

- Todos os valores são negativos (dias relativos à aplicação — semântica do Home Credit)
- `DAYS_ENTRY_PAYMENT` pode ser menor que `DAYS_INSTALMENT` (pagamento antecipado)
- **Não criar feature de atraso** (reservado para Gold)

### 6. Campos Financeiros

| Campo | Min | Max | Média | Mediana | NULLs | Neg |
|-------|-----|-----|-------|---------|-------|-----|
| `AMT_INSTALMENT` | 0 | 3.771.488 | 17.051 | 8.884 | 0 | 0 |
| `AMT_PAYMENT` | 0 | 3.771.488 | 17.238 | 8.125 | 2.905 | 0 |

- Sem valores negativos ✅
- 290 parcelas com valor zero (anomalia — flag criada)
- 1.440 pagamentos com valor zero (flag criada)

### 7. Consistência de Pagamentos

| Regra | Afetados | % |
|-------|----------|---|
| AMT_INSTALMENT = 0 | 290 | 0,002% |
| AMT_PAYMENT = 0 | 1.440 | 0,011% |
| AMT_PAYMENT NULL | 2.905 | 0,021% |
| DAYS_ENTRY_PAYMENT NULL | 2.905 | 0,021% |
| AMT_PAYMENT > AMT_INSTALMENT | ~varios | ~vários |

- Pagamento maior que parcela é **legítimo** (pagamento antecipado, ajustes, consolidação)
- Nenhum registro removido
- Features de atraso/ratio **não criadas** (reservadas para Gold)

### 8. Flags de Validação Criadas

| Flag | Descrição | Registros |
|------|-----------|-----------|
| `FLAG_PAYMENT_MISSING` | AMT_PAYMENT IS NULL | 2.905 |
| `FLAG_AMT_INSTALMENT_ZERO` | AMT_INSTALMENT = 0 | 290 |
| `FLAG_AMT_PAYMENT_ZERO` | AMT_PAYMENT = 0 | 1.440 |

### 9. Integridade Referencial
- `installments_payments.SK_ID_PREV` → `silver.previous_application.SK_ID_PREV`
- `installments_payments.SK_ID_CURR` → `silver.application_train/test.SK_ID_CURR`
- Nenhum registro removido por falta de correspondência

### 10. Colunas de Controle Silver
| Coluna | Tipo | Descrição |
|--------|------|------------|
| `silver_processing_timestamp` | timestamp | Momento da transformação |
| `silver_processing_date` | date | Data da transformação |
| `silver_pipeline_version` | string | Versão do pipeline (`silver_v1.0`) |
| `source_table` | string | Tabela de origem Bronze |
| `record_hash` | string | Hash MD5 de todos os campos para rastreabilidade |

### 11. Schema da Tabela Silver
- **Bronze**: 10 colunas (8 dados + 2 metadados)
- **Silver**: 16 colunas (8 dados + 3 flags + 5 controle - 2 metadados removidos)
- Nenhuma coluna original foi modificada em tipo ou semântica
- **Nenhuma feature de ML foi criada** (payment_delay, late_payment_flag, payment_ratio, etc.)